# Clase 190 — Uplift modeling y DiD (difference-in-differences)

Dos técnicas causales de industria con datos observacionales/panel: **DiD** (evolución antes/después en tratado vs control, vía OLS con interacción) y **uplift modeling** (predecir a quién conviene tratar, CATE individual con T-learner + Qini).

Requiere: `numpy`, `pandas`, `statsmodels`, `scikit-learn`, `matplotlib`.

## 1. DiD con OLS

`Y = β₀ + β₁·tratado + β₂·post + β₃·(tratado×post) + ε`. El coeficiente `β₃` es el efecto causal bajo **parallel trends**.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

n = 2000
treated = rng.integers(0, 2, n)
post = rng.integers(0, 2, n)
true_effect = 2.5
Y = 10 + 1.5*treated + 3.0*post + true_effect*(treated*post) + rng.normal(0, 2, n)
df = pd.DataFrame({"Y": Y, "treated": treated, "post": post})
m = smf.ols("Y ~ treated * post", data=df).fit()
beta3 = m.params["treated:post"]
ci = m.conf_int().loc["treated:post"]
print(f"DiD β3 = {beta3:.3f}  IC95%=({ci[0]:.3f}, {ci[1]:.3f})  verdadero={true_effect}")
assert abs(beta3 - true_effect) < 0.5

## 2. Event study: chequear parallel trends

Con varios períodos, la diferencia tratado-control debe ser ≈ 0 **antes** del tratamiento (pre-tendencias planas) y saltar después.

In [ ]:
periods = np.arange(-5, 6)            # -5..-1 pre, 0..5 post
unit_treated = np.repeat([0, 1], 200)
rows = []
for u, tr in enumerate(unit_treated):
    ai = rng.normal(0, 1)             # efecto fijo de unidad
    for tp in periods:
        eff = true_effect if (tr == 1 and tp >= 0) else 0.0
        rows.append((u, tr, tp, 5 + 0.2*tp + ai + eff + rng.normal(0, 1)))
pan = pd.DataFrame(rows, columns=["unit", "treated", "t", "y"])
diff = pan[pan.treated == 1].groupby("t").y.mean() - pan[pan.treated == 0].groupby("t").y.mean()
pre = diff.loc[-5:-1].abs().mean()
print(f"|dif| media pre-tratamiento = {pre:.2f}  (≈0 => parallel trends plausible)")
assert pre < 0.5

plt.figure(figsize=(7, 4))
plt.plot(diff.index, diff.values, "o-")
plt.axvline(-0.5, color="k", ls="--", label="inicio tratamiento")
plt.axhline(0, color="gray", lw=0.8)
plt.xlabel("período relativo"); plt.ylabel("dif. tratado - control")
plt.title("Event study: pre≈0, salto en post"); plt.legend()
plt.tight_layout(); plt.show()

## 3. Uplift con T-learner

Dos modelos separados (`μ₁` para tratados, `μ₀` para control); el uplift es `μ₁(x) - μ₀(x)`. Simulamos un efecto que crece con la feature `x`.

In [ ]:
N = 8000
x = rng.uniform(0, 1, N)
t = rng.integers(0, 2, N)
p = (0.15 + 0.2*x) + (0.5*x) * t          # uplift verdadero = 0.5*x
y = (rng.uniform(0, 1, N) < p).astype(int)
data = pd.DataFrame({"x": x, "t": t, "y": y})
Xt = data[["x"]].values

m1 = RandomForestClassifier(n_estimators=200, min_samples_leaf=50, random_state=42).fit(Xt[t == 1], y[t == 1])
m0 = RandomForestClassifier(n_estimators=200, min_samples_leaf=50, random_state=42).fit(Xt[t == 0], y[t == 0])
grid = np.linspace(0, 1, 100).reshape(-1, 1)
uplift_pred = m1.predict_proba(grid)[:, 1] - m0.predict_proba(grid)[:, 1]
corr = np.corrcoef(grid.ravel(), uplift_pred)[0, 1]
print(f"correlación(uplift_pred, x) = {corr:.3f}  (>0: capta que el uplift crece con x)")
assert corr > 0.5

## 4. Qini curve

Ordenamos a los individuos por uplift predicho y acumulamos la ganancia incremental frente a tratar al azar.

In [ ]:
data["uplift_hat"] = m1.predict_proba(Xt)[:, 1] - m0.predict_proba(Xt)[:, 1]
o = data.sort_values("uplift_hat", ascending=False).reset_index(drop=True)
cum_t  = (o.t == 1).cumsum().values
cum_c  = (o.t == 0).cumsum().values
cum_yt = (o.y * (o.t == 1)).cumsum().values
cum_yc = (o.y * (o.t == 0)).cumsum().values
ratio = np.divide(cum_t, np.maximum(cum_c, 1))
qini = cum_yt - cum_yc * ratio
k = np.arange(1, len(o) + 1)

plt.figure(figsize=(7, 4))
plt.plot(k, qini, label="modelo uplift")
plt.plot([0, len(o)], [0, qini[-1]], "k--", label="aleatorio")
plt.xlabel("# individuos tratados (ordenados por uplift)")
plt.ylabel("ganancia incremental"); plt.legend(); plt.title("Qini curve")
plt.tight_layout(); plt.show()
print(f"ganancia final acumulada = {qini[-1]:.1f}")

## 5. DiD vs diferencia ingenua

La diferencia post ingenua mezcla el efecto con la diferencia basal entre grupos; el DiD la aísla.

In [ ]:
naive_post = (df[(df.treated == 1) & (df.post == 1)].Y.mean()
              - df[(df.treated == 0) & (df.post == 1)].Y.mean())
print(f"diferencia ingenua post = {naive_post:.3f}  (mezcla efecto + dif. basal)")
print(f"DiD β3                  = {beta3:.3f}  (aísla el efecto causal ≈ 2.5)")
assert naive_post > beta3

## Ejercicios

1. Violá parallel trends dándole al control una tendencia distinta en el pre y observá cómo el event study lo delata.
2. Entrená un X-learner simple (ponderá los pseudo-efectos por propensity) y compará su uplift con el T-learner.
3. Evaluá el modelo con `uplift@k` (ganancia al tratar el top 20 %) además de la Qini completa.

## Conclusiones

- DiD estima el efecto como el coeficiente de la interacción `tratado×post`; su validez depende de **parallel trends**.
- Verificá parallel trends con un event study (pre-tendencias planas) antes de creerle al `β₃`.
- El uplift es CATE individual: se evalúa con Qini / uplift@k, **no** con AUC de clasificación.
- T/S/X-learner y causal forest estiman heterogeneidad del efecto para decidir a quién tratar.